# Moeve × Classiq: starter notebook

This notebook shows how to use the challenge files. It does **not** solve anything.

* `data/easy.json` (Levels 0 and 1), `data/intermediate.json` (Level 2), `data/full_on.json` (Level 3a): plant data and both encodings
* `refinery.py`: the plant evaluator (`evaluate`, `decode`, `evaluate_bits`), the HDS model (`hds_outlet`) and the mixed-integer LP (`complete_plan`)

Read the [challenge README](README.md) first.

In [ ]:
import json
import refinery as R

inst = R.load_instance("easy")
print(inst["name"])
for note in inst["notes"]:
    print("-", note)

## 1. The plant

In [ ]:
import pandas as pd

crudes = pd.DataFrame({c: {"price EUR/t": d["price_eur_t"], "availability kt": d["availability_kt"],
                           "ADO core S %": d["qualities"]["ado_core"]["sulphur_pct"],
                           "ADO core cetane": d["qualities"]["ado_core"]["cetane"],
                           "swing cut (A/R) S %": d["qualities"]["ar_swing"]["sulphur_pct"],
                           "CDUs": ", ".join(d["allowed_cdus"])}
                       for c, d in inst["crudes"].items()}).T
display(crudes)
display(pd.DataFrame(inst["products"]).T)
print("import:", inst["import_component"])
print("netbacks:", inst["prices"])

## 2. The plan as qubits (two encodings)

Each variable has `bits` qubits. Integer level `k` maps to `values[k]`. A bitstring lists the variables in this order, most-significant bit first.

* `discrete`: every decision on qubits (pool split and import in intervals)
* `mixed_integer`: only the operating mode on qubits; pool split and import are continuous (section 7)

In [ ]:
pd.DataFrame([{"variable": v["name"], "bits": v["bits"], "values": v["values"]} for v in R.decision_variables(inst)])

In [ ]:
for mode in ["discrete", "mixed_integer"]:
    n = R.num_qubits(inst, mode)
    print(f"{mode:14s} {n} qubits -> {2**n:,} combinations")

## 3. Evaluate a plan in physical units

In [ ]:
plan = R.Plan(
    crude_kt={"A": 120, "C": 0},
    alpha={"A": 0.5, "C": 0.5},   # half of the ADO/Residue swing cut goes up into ADO
    premium_share=0.0,            # nothing to Premium Diesel
    standard_share=1.0,           # whole pool to Standard Diesel
    import_kt=0,
)
res = R.evaluate(inst, plan)
print(res)
pd.Series(res.breakdown).round(1)

In [ ]:
pd.Series(res.streams).round(3)

## 4. Evaluate a bitstring
Buy everything, highest cut temperatures, maximum premium and import. Look at which constraints break.

In [ ]:
bits = "11" "11" "11" "11" "11" "00" "11"   # crude_A, crude_C, cut_temp_A, cut_temp_C, premium_share, hydrocracker_share, import
plan = R.decode(inst, bits)
print(plan)
res = R.evaluate_bits(inst, bits)
print(res)
res.violations

## 5. Count your evaluations

For the benchmark you need to know how many times a method looked at a plan. A simple wrapper:

In [ ]:
class CountingEvaluator:
    def __init__(self, inst):
        self.inst, self.calls = inst, 0

    def __call__(self, bits: str):
        self.calls += 1
        return R.evaluate_bits(self.inst, bits)

f = CountingEvaluator(inst)
f("01" "00" "10" "10" "00" "00" "00")
print("evaluations so far:", f.calls)

## 6. The Diesel HDS (Level 2 and up)

The `intermediate` instance (calibrated by Moeve: sulphurous crudes, crude C on CDU-1 or CDU-2) adds a hydrotreater between the pool and blending. Its outlet sulphur, yield and cost are nonlinear in feed sulphur, load and severity (Moeve's Appendix B5). Same plan, four severities: bypass and z = 0 break the sulphur specs; z = 0.5 and z = 1 fix them, at a different cost. The HDS inlet sulphur is computed from the crudes, cut temperatures and swing-cut routing.

In [ ]:
inter = R.load_instance("intermediate")
print(inter["hds"])
rows = []
for z in [None, 0.0, 0.5, 1.0]:
    plan = R.Plan(crude_kt={"A": 180, "C": 80}, alpha={"A": 1/3, "C": 0.0}, cdu={"C": "CDU-2"},
                  premium_share=1/6, standard_share=5/6, import_kt=45, hds_severity=z)
    r = R.evaluate(inter, plan)
    rows.append({"severity": "bypass" if z is None else z, "margin kEUR": round(r.margin, 1), "feasible": r.feasible,
                 "pool S % after HDS": round(r.streams["pool_sulphur_pct"], 3), "HDS load": round(r.streams.get("hds_load", 0), 2),
                 "HDS cost kEUR": round(-r.breakdown["hds_cost"], 1), "violations": list(r.violations)})
pd.DataFrame(rows)

In [ ]:
# The unit model on its own: outlet sulphur vs. severity (note the minimum near z = 0.64)
for z in [0.0, 0.25, 0.5, 0.64, 0.75, 1.0]:
    s_out, eta = R.hds_outlet(sulphur_in_pct=0.5, load=0.8, severity=z)
    print(f"z={z:.2f}  S_out={s_out:.4f} %  yield={eta:.4f}")

## 7. Mixed-integer: integer operating mode, continuous blending

Fix the operating mode (crudes, temperatures, CDUs, severity) as a bitstring of the `mixed_integer` encoding. `complete_plan` then solves the small LP for the best continuous pool split and import, and returns the plan and its evaluation.

In [ ]:
print([v["name"] for v in R.decision_variables(inter, "mixed_integer")])
mode_bits = "01" "10" "11" "00" "1" "10"   # crude_A, crude_C, cut_temp_A, cut_temp_C, cdu_C, hds_severity
plan, res = R.complete_plan(inter, mode_bits)
print(plan)
print(res)

In [ ]:
# complete_plan also takes a Plan, so crude amounts and cut temperatures can be continuous too
plan, res = R.complete_plan(inter, R.Plan(crude_kt={"A": 75.0, "C": 150.0}, alpha={"A": 0.9, "C": 0.1}, cdu={"C": "CDU-2"}, hds_severity=0.5))
print(round(plan.premium_share, 3), round(plan.standard_share, 3), round(plan.import_kt, 1), res)

## 8. Your turn

**Level 0 · Baselines** (`easy`): exact discrete optimum, continuous optimum and discretization gap, a heuristic and its number of evaluations.

**Level 1 · Easy** (`easy`, 14 qubits): encoding and cost function / oracle in Qmod (`phase(-cost(v), gamma)`; constraints gated with `control(...)` or as penalties); QAOA and/or Grover search; queries, quality and resources vs. Level 0.

**Level 2 · Intermediate** (`intermediate`): 2a fully discrete (17 qubits) and/or 2b mixed-integer (11 qubits + `complete_plan`); compare them.

**Level 3 · Full-on** (`full_on`, then Moeve's full case): scale up, estimate resources, place the full case on the hardware roadmap.

Each level has a *Go further* list in the README. Useful starting points are listed at the end of the README.